## Speech to Text processing

#### Import all required libraries

In [ ]:
import pandas as pd
import webbrowser, os
import json
import boto3
import re
import sagemaker
import time
import uuid
from sagemaker import get_execution_role
from pprint import pprint

#### Define variables that we need

In [ ]:
s3 = boto3.client('s3')
transcribe = boto3.client('transcribe')
role = get_execution_role()
print(role)
bucket = 'a2i-experiments'
prefix = 'nlp-aws-book4/chapter-03'

### Download YouTube video of the author's AI/ML conference

In [ ]:
!pip install pytube --quiet

In [ ]:
from pytube import YouTube
yt = YouTube('https://www.youtube.com/watch?v=vBtxjXjr_HA')

In [ ]:
# Create a new directory for storing our audio file and other datasets
main_dir = '/root/cuttingedge-nlp-aws/Chapter-03/'
new_dir = 'raw-audio/'
path = os.path.join(main_dir, new_dir)
if not os.path.exists(path):
    os.mkdir(path)

In [ ]:
# change working directory to chapter 3 folder
# Make sure you change this path to whatever directory your audio files are in
path = '/root/cuttingedge-nlp-aws/Chapter-03/raw-audio/'
os.chdir(path)

In [ ]:
yt_file = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first().download()
print(yt_file)

In [ ]:
old_file = yt_file.split('/')[5]
new_file = 'speech.mp4'

In [ ]:
os.rename(old_file,new_file)

### Use Amazon Transcribe to create text transcript

In [ ]:
mg_audio_path = '/root/cuttingedge-nlp-aws/Chapter-03/'
mg_audio_file = 'raw-audio/'+new_file

In [ ]:
# First let us upload the MP4 file to an Amazon S3 bucket
s3.upload_file(mg_audio_path+mg_audio_file,bucket,prefix+'/'+mg_audio_file)
s3_upload_path = 's3://'+bucket+'/'+prefix+'/'+mg_audio_file
print("uploaded successfully to: " + s3_upload_path)

Now go to your [Amazon S3 console](https://s3.console.aws.amazon.com/s3/buckets?region=us-east-1) to double check the MP4 file was successfully copied to the S3 URI printed above

#### Run transcription using Amazon Transcribe

In [ ]:
job_name = 'nlp-chapter3-'+str(uuid.uuid4())
lang_code = 'en-US' # English US. If you dont care for the language code, you can include IdentifyLanguage parameter at transcription time
job_uri = s3_upload_path

In [ ]:
transcribe.start_transcription_job(
            TranscriptionJobName=job_name,
            LanguageCode=lang_code,
            Media={"MediaFileUri": job_uri}
            )
time.sleep(2)        
print(transcribe.get_transcription_job(TranscriptionJobName=job_name)['TranscriptionJob']['TranscriptionJobStatus'])

This will take approximately 1 to 3 minutes to complete. You can check the transcription job status in the [Amazon Transcribe console](https://console.aws.amazon.com/transcribe/home?region=us-east-1#jobs). Once the job completes, you can retrieve the transcription text we will need for this chapter to try out the **Spacy, TextBlob and GenSim** libraries

#### Retrieve transcription text
Execute the following steps after your transcription job successfully completes or you will get errors. Try again after job completion to proceed.

In [ ]:
# Download the transcript from Amazon Transcribe's internal S3 bucket and upload it to our S3 bucket
import urllib3
response = transcribe.get_transcription_job(
    TranscriptionJobName=job_name 
)
file = response['TranscriptionJob']['Transcript']['TranscriptFileUri']
http = urllib3.PoolManager()
data = http.request('GET', file)
results = json.loads(data.data.decode('utf-8'))
transcript = results["results"]["transcripts"]
outfile = 'transcript.txt'
with open(outfile, 'w') as out:
    out.write(transcript[0]['transcript'])
s3.upload_file(outfile,bucket,prefix+'/transcript/'+outfile)
s3_transcript_path = f's3://{bucket}/{prefix}/transcript/{outfile}'
print("Transcript uploaded to: " + s3_transcript_path)

In [ ]:
#### Convert to string and load into a variable
# Get the youtube transcript from the S3 bucket using Pandas
df = pd.read_table(s3_transcript_path,header=None)
# Convert to string
in_text = df.iloc[0].values[0]
# print just a substring for brevity's sake
print(in_text[:1001])

## Working with SPACY

In [ ]:
# Update pip
!pip install -U pip --quiet

In [ ]:
# Install the Spacy library
!pip install -U spacy --quiet

In [ ]:
# Download the English medium size model for our needs
!python -m spacy download en_core_web_md

In [ ]:
# Now load the model so we can use it
import spacy
model = spacy.load('en_core_web_md')
# Print the pre-added list of tasks the model performs on your data
# This is good to know currently, but we will use this later when we run text classification using Spacy
model.pipe_names

In [ ]:
# Lets get Spacy's interpretation of our input text, again selecting a few lines only for brevity
spacy_text = model(in_text[:1001])

### Spacy - Word tokenization
Tokenization is the first step in NLP feature engineering. You tokenize to get a collection of words from your text document and use it for techniques such as CBOW or Skipgram to predict neighboring words

In [ ]:
# Printing the first ten words here
for token in spacy_text[:10]:
    print("Word: " + token.text)

### Spacy - Text processing

In [ ]:
# case conversion
upper = str(spacy_text[:10]).upper()
print("Original sentence: " + upper)

lower = ''
for token in model(upper):
    lower += token.lower_ + " "  
print("Lower case version: " + lower)

In [ ]:
# Does string contain alphabetic charcacters
for token in spacy_text[:5]:
    print(token.text + " ----> " + str(token.is_alpha))

In [ ]:
# Does string have punctuations
for token in spacy_text[20:25]:
    print(token.text+ " ----> " + str(token.is_punct))

In [ ]:
# are there white spaces
for token in spacy_text[20:30]:
    print(token.text + " ----> " + str(token.is_space))

In [ ]:
# does the token start or end a sentence
sent = ''
for token in spacy_text[25:30]:
    sent += token.text + ' '
    print("Word: " + sent)
    print("Start of sentence?: " + str(token.is_sent_start))
    print("End of sentence?: " + str(token.is_sent_end))

### Spacy - Visualize Sentences

In [ ]:
from spacy import displacy
# create sentences from our text - only selecting the first 100 characters for brevity
sentences = list(spacy_text[:50].sents)
displacy.render(sentences, style="dep")

### Spacy - Generating Word Vectors

In [ ]:
for token in spacy_text[:5]:
    print("Word: " + token.text, "\n Word Vector: " + str(token.vector_norm))

### Spacy - text language

In [ ]:
for token in spacy_text[:5]:
    print(token.text, token.lang_)

### Spacy - Grammatical Syntax Prediction - POS tagging

The list of higher level part of speech is: https://universaldependencies.org/u/pos/

The list of lower level part of speech is: https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html

In [ ]:
for token in spacy_text[:5]:
    print("Word: " + token.text, "\n Higher level part of speech: " + str(token.pos_), "\n Lower level part of speech: " + str(token.tag_))

### Spacy - Grammatical relationship between words - Dependency Parsing
Dependency parsing and visualization

In [ ]:
for token in spacy_text[:5]:
    print("Word: " + token.text, "\n Dependency parsed tags: " + str(token.dep_))

### Spacy - Sentence Similarity

In [ ]:
print("Sentence One: " + in_text[250:500] + '\n')
sentence_one = model(in_text[250:500])
print("Sentence Two: " + in_text[750:1000] + '\n')
sentence_two = model(in_text[750:1000])
sent_similarity = sentence_one.similarity(sentence_two)
print("Sentence similarity is: " + str(round(float(sent_similarity)*100,0)) + '%')

### Spacy - Named Entity Recognition

In [ ]:
# We will consider text from position 1001 onwards
# You can see some incorrect entity recognition - compare this with Comprehend
print(in_text[1000:2000])
ent_text = model(in_text[2000:])
for entity in ent_text.ents:
    print(entity.text, entity.start_char, entity.end_char, entity.label_)

### Spacy - Visualizing Named Entity Recognition

In [ ]:
from spacy import displacy
displacy.render(ent_text, style="ent")

### Spacy - Lemmatizing words

In [ ]:
for token in spacy_text[120:130]:
    print("The word: - " + str(token.text), "- The Lemma for the word: " + str(token.lemma_))

### Spacy - Remove Stop Words and Punctuations

In [ ]:
# Removing stop words
stop_removed = ''
for token in spacy_text:
    if token.is_stop==False: # not a stop word
        if token.is_punct==False:
            stop_removed += str(token) + ' '
print("Text with stop words and punctuation removed: " + str(stop_removed))

## Working with TEXTBLOB
TextBlob is a python NLP package built on top of NLTK we reviewed in Chapter 2. It provides additional features composed from core NLTK features

In [ ]:
# install TextBlob and download the text corpora
!pip install -U textblob
!python -m textblob.download_corpora

### Create TextBlobs

In [ ]:
from textblob import TextBlob
sm_blob = TextBlob(in_text[:100])
md_blob = TextBlob(in_text[:500])
lg_blob = TextBlob(in_text[:1000])
xl_blob = TextBlob(in_text)

### TextBlob - Grammatical Syntax - POS tagging

In [ ]:
# Use the small text blob
sm_blob.tags

### TextBlob - Grammatical relationships - Dependency Parsing

In [ ]:
# using the small text blob for dependency parsing
sm_blob.parse()

### TextBlob - Extract Noun phrases

In [ ]:
# using the large blob for phrase extraction
lg_blob.noun_phrases

### TextBlob - Detect sentiment

In [ ]:
print(sm_blob)
# Get sentiment and polarity using the small blob
sm_blob.sentiment

### TextBlob - Tokenize Words

In [ ]:
# With textblob you can tokenize docs into both sentences and words easily
md_blob.sentences

In [ ]:
md_blob.words

### TextBlob - Word Inflection

In [ ]:
# Inflection refers to singular and plural representation of words
md_blob.sentences[1].words[6].pluralize()

In [ ]:
md_blob.sentences[0].words[4].singularize()

### TextBlob - Lemmmatization

In [ ]:
# Lemmatizer needs an additional NLTK resource
import nltk
nltk.download('omw-1.4')

In [ ]:
# Lemmatize all words in Sentence 3 of the xl blob
from textblob import Word
for word in xl_blob.sentences[4].words:
    print(word + " ---> " + Word(word).lemmatize())

### TextBlob - Extract semantically similar words
To get semantically similar words from a corpus we can use the synsets feature of WordNet (https://wordnet.princeton.edu/)

In [ ]:
# Get synsets for a sentence of type Noun, Verb, adjective and adverb
from textblob import Word
from textblob.wordnet import VERB
from textblob.wordnet import NOUN
from textblob.wordnet import ADJ
from textblob.wordnet import ADV

print("Semantically similar NOUNS are")
for word in md_blob.sentences[2].words:
    if len(Word(word).get_synsets(pos=NOUN)) > 0:
        print(word)
        print(" ---> " + str(Word(word).get_synsets(pos=NOUN)))
print('\n')
print("Semantically similar VERBS are")
for word in md_blob.sentences[2].words:
    if len(Word(word).get_synsets(pos=VERB)) > 0:
        print(word)
        print(" ---> " + str(Word(word).get_synsets(pos=VERB)))
print('\n')
print("Semantically similar ADJECTIVES are")
for word in md_blob.sentences[2].words:
    if len(Word(word).get_synsets(pos=ADJ)) > 0:
        print(word)
        print(" ---> " + str(Word(word).get_synsets(pos=ADJ)))
print('\n')
print("Semantically similar ADVERBS are")
for word in md_blob.sentences[2].words:
    if len(Word(word).get_synsets(pos=ADV)) > 0:
        print(word)
        print(" ---> " + str(Word(word).get_synsets(pos=ADV)))
print('\n')

### TextBlob - Check and correct spelling

In [ ]:
# Check and report what the correct spelling is
wrong_words = ['upples','worangis','mengoas','fruitas','carrat','vegitables']
for word in wrong_words:
    print("Original word is: " + word)
    print(Word(word).spellcheck())

In [ ]:
# Correct the spelling for the sentence as a whole
wrong_words = "upples worangis mengoes ari fruits carrat ari vegitables"
wrong_blob = TextBlob(wrong_words)
print(wrong_blob.correct())

### TextBlob - Word and Noun phrase counts

In [ ]:
# word counts - take the large blob
xl_blob.words.count('documents')

In [ ]:
#noun phrase counts
print("Count of Noun Phrase document workflow is ---> " + str(xl_blob.noun_phrases.count("document workflow")))

### TextBlob - Basic text processing

In [ ]:
# Get substrings using Python syntax
sm_blob[:10]

In [ ]:
#you can change case
sm_blob.upper()

In [ ]:
sm_blob.lower()

In [ ]:
# you can split the string
sm_blob.split(' ')

In [ ]:
# you can find the index for a word
sm_blob.find('documents')

### TextBlob - Finding N grams

In [ ]:
# We will get bigrams from the small blob
sm_blob.ngrams(n=2)

### TextBlob - Sentence indexes

In [ ]:
for sentence in md_blob.sentences:
    print(sentence + " Begin Offset: " + str(sentence.start) + " End Offset: " + str(sentence.end))

### Text classification with TextBlob
Setting up a classifer in TextBlob is much more simpler than in SpaCy thanks to pre-built packages instead of having to fiddle with pipeline tasks and worrying about data formats, creating Example data structures and so on

### Amazon reviews dataset

In [ ]:
# change working directory to chapter 3 folder
# Make sure you change this path to whatever directory your training CSV file is in
path = '/root/cuttingedge-nlp-aws/Chapter-03/'
os.chdir(path)

In [ ]:
raw_df = pd.read_csv('text-classifier-train.csv')
# change the label to the last column
raw_df = raw_df[['review','class']]
raw_df.head()

In [ ]:
# Create train and test datasets
train_df = raw_df.iloc[:950,:]
test_df = raw_df.iloc[951:1000,:]

In [ ]:
train_df.dtypes

In [ ]:
# Now convert train data to CSV file
train_df.to_csv('train.csv',index=False, header=False)

#### Naive Bayes Classifier

In [ ]:
from textblob.classifiers import NaiveBayesClassifier
with open('train.csv', 'r') as train_file:
    nb_model = NaiveBayesClassifier(train_file, format='csv')

In [ ]:
pred_df = pd.DataFrame()
for i, r in test_df.iterrows():
    predicted = nb_model.classify(r['review'])
    pred_df.loc[i,'review'] = r['review']
    pred_df.loc[i,'class'] = int(r['class'])
    pred_df.loc[i,'nb_prediction'] = round(int(predicted),0)
pred_df.head(10)

In [ ]:
pred_df.dtypes

In [ ]:
# Calculate F1 score for Naive Bayes model using Scikit Learn Metrics
from sklearn.metrics import f1_score
labels = pred_df['class'].values.tolist()
nb_preds = pred_df['nb_prediction'].values.tolist()
print(round(f1_score(labels, nb_preds, average='weighted')*100,0))

#### Positive Naive Bayes Classifier
The prior positive probability is provided as 90% because of our unbalanced class distribution

In [ ]:
from textblob.classifiers import PositiveNaiveBayesClassifier
good_reviews = []
for i,r in train_df.iterrows():
    if r['class'] == 1:
        good_reviews.append(r['review'])

pnb_model = PositiveNaiveBayesClassifier(positive_set=good_reviews, unlabeled_set=test_df['review'].values.tolist(), positive_prob_prior=0.9)

In [ ]:
for i, r in test_df.iterrows():
    predicted = pnb_model.classify(r['review'])
    if predicted == True:
        pred_df.loc[i,'pnb_prediction'] = 1
    else:
        pred_df.loc[i,'pnb_prediction'] = 0
        
pred_df.head(10)

In [ ]:
# Calculate F1 score for Positive Naive Bayes model using Scikit Learn Metrics
from sklearn.metrics import f1_score
labels = pred_df['class'].values.tolist()
pnb_preds = pred_df['pnb_prediction'].values.tolist()
print(round(f1_score(labels, pnb_preds, average='weighted')*100,0))

## Working with GENSIM
The 3rd python library we review today Gensim can do many of the NLP tasks of its counterparts and in addition can be used for NLP modelling needs such as topic modelling, TFIDF models, sentence similarity and more. For a full list of NLP techniques Gensim supports, please refer to - https://radimrehurek.com/gensim/index.html

In [ ]:
!pip install -U gensim

### Create vector representations

In [ ]:
# Create Spacy representation from a TextBlob blob
spacy_text = model(str(xl_blob))

In [ ]:
# Next we will use Spacy to remove stop words from this
stop_removed = ''
for token in spacy_text:
    if token.is_stop==False: # not a stop word
        stop_removed += str(token) + ' '
print("Text with stop words removed: " + str(stop_removed))

In [ ]:
# First create an empty list for sentences
sentence_list = []


# we will use TextBlob for this
gensim_blob = TextBlob(stop_removed)

# only add words that are repeated for each sentence
for sentence in gensim_blob.sentences:
    word_list = []
    for word in sentence.words:
        if gensim_blob.words.count(word) > 1:
            word_list.append(str(word).lower())
    if len(word_list) > 1:
        sentence_list.append(word_list)

In [ ]:
sentence_list

In [ ]:
# Now create a Gensim dictionary object
from gensim import corpora

gensim_dict = corpora.Dictionary(sentence_list)
print(gensim_dict)

Since there are 74 unique tokens, Gensim will build a 74 dimension vector to interpret our input document. A document can be a group of words, a sentence, a paragraph or a page or pages. Each token will be assigned a unique ID (token ID) to represent each of the 74 dimensions, and in the Bag of Words model, Gensim will create a sparse vector of our document using the token ID and the token frequency in the document.

In [ ]:
# get token IDs
gensim_dict.token2id

In [ ]:
# Testing Gensim vectorization
input_text = 'banks have a lot of documents that will benefit from automated insights'
gensim_vectors = gensim_dict.doc2bow(input_text.lower().split())
print(gensim_vectors)

In the results, the first entry for each pair corresponds to the token ID for the words Gensim found in the dictionary, and the second entry corresponds to the count of this word in the input document

In [ ]:
# Convert our XL blob into Gensim vectors
gensim_vectors = []
for sentence in sentence_list:
    gensim_vectors.append(gensim_dict.doc2bow(sentence))
                          
print(gensim_vectors)

### Gensim Topic Modelling - TF-IDF

In [ ]:
from gensim import models

# Create the TF-IDF model
tf_model = models.TfidfModel(gensim_vectors)

# testing the model
input_tokens = input_text.lower().split()
tf_model[gensim_dict.doc2bow(input_tokens)]

The first entry is the tokenID from our 74 dimensional vector and the second entry is the TF-IDF weight that indicates the importance of each document in the overall document corpus. In this example tokenID 8 which stands for automated is the most important word in the input document, which makes a lot of sense. This simple test clearly shows how easy it is to use TF-IDF to do something like topic modelling

#### Semantic matching using TF-IDF

In [ ]:
# Using our earlier input text let us see which of our sentences or docs closely match that
from gensim import similarities

gensim_search_idx = similarities.SparseMatrixSimilarity(tf_model[gensim_vectors], num_features=74)

query_vectors = gensim_dict.doc2bow(input_tokens)

match_results = gensim_search_idx[tf_model[query_vectors]]
list(enumerate(match_results))

In [ ]:
print(max(match_results))

The results show that sentence 3 has the most similarity with the input text. Lets print them to see why.

In [ ]:
print("Input sentence is: " + input_text)
for i, sentence in enumerate(xl_blob.sentences):
    if i == 4:
        print("Similar sentence in our XL blob identified by the TF-IDF model: " + str(sentence))

For enterprise scale use cases for semantich matching, you can use **Amazon Kendra**, a managed service for semantic matching and NLP powered query capabilities with support for 100+ data connectors, virtually unlimited scale, superior performance and reliability.

### Gensim Topic Modelling - Latent Semantic Indexing
Create the LSI model and prints 3 topics from our XL blob that LSI thinks is semantically related. 

In [ ]:
from gensim import models
# Create the LSI model
lsi_model = models.LsiModel(gensim_vectors, id2word=gensim_dict, num_topics=3)
# Ask the LSI model to detect 3 topics for the XL blob
lsi_vectors = lsi_model[gensim_vectors]
# print the topics detected
lsi_list = lsi_model.print_topics(3)
print(lsi_list)

### Gensim Topic Modelling - Latent Dirchlet Allocation

In [ ]:
from gensim import models
# Create the LSI model
lda_model = models.LdaModel(gensim_vectors, id2word=gensim_dict, num_topics=5)
# Ask the LSI model to detect 5 topics for the XL blob
lda_model[gensim_vectors]
# print the topics detected
lda_model.print_topics(5)

## END OF NOTEBOOK 
Please follow instructions from Chapter 3 in the book to proceed